In [ ]:
# Import Libraries
import sys
from pathlib import Path

project_root = Path.cwd().parent

if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

In [ ]:
import random
from services import league_services
from datetime import datetime

_RANKED_LEAGUES    = frozenset({"IRON", "BRONZE", "SILVER", "GOLD", "PLATINUM", "EMERALD", "DIAMOND"})
_DIVISIONS         = frozenset({"I", "II", "III", "IV"})

servers = ["kr", "na1", "euw1", "br1", "eun1", "jp1", "la1"]

for i in range(3000):

    random_queue = random.choices(["RANKED_SOLO_5x5", "RANKED_FLEX_SR"], weights=[80, 20], k=1)[0]
    random_league = random.choice(list(_RANKED_LEAGUES))
    random_division = random.choice(list(_DIVISIONS))
    random_page = random.randint(1, 15)
    server = random.choice(servers)

    entries = league_services.fetch_ranked_league_entries(server, random_queue, random_division, random_league, random_page)
    print(datetime.now().strftime("%d/%m/%Y, %H:%M:%S"), i, server, random_queue, random_league, random_division, random_page, len(entries))

    x = 0
    for entry in entries:

        puuid = entry["puuid"]
        try:
            summoner_data = league_services.register_summoner(server, puuid=puuid)
            ranked_data = league_services.capture_summoner_ranks(f"{summoner_data["summoner_name"]}#{summoner_data["tag_line"]}", server)
            matchlist = league_services.fetch_recent_match_ids(server, puuid, count=2)
        except Exception as e:
            print(f"Fail {e}")

        for match_id in matchlist:

            status = league_services.store_match(match_id)
            players = league_services.get_match_participants(match_id)

        x = x + 1
        if x == 3:
            break